# Free-sample pilot: risk ratio and odds ratio

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Risk Ratio and Odds Ratio** · a worked example from the StatNex teaching guide

**The story.** A snack brand ran a small in-store pilot: 11 shoppers were offered a free sample and 11 were not, and staff noted who bought a pack. This example measures how big the difference is, not just whether there is one.

**The question.** How much more likely is a shopper to buy after a free sample?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/risk_ratio_odds_ratio.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Group,Bought,Did not buy,Customers
Free sample,8.0,3.0,11.0
No sample,2.0,9.0,11.0
Total,10.0,12.0,22.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import contingency
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.proportion import confint_proportions_2indep

pd.set_option("display.width", 120)

## The data

In [ ]:
# A 2 x 2 table of counts. The first row is the exposed group (the new drug), the
# first column the outcome being counted (an effect).
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Bought", "Did not buy"]
table = data.set_index("Group")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()
counts = table.to_numpy(float)
(a, b), (c, d) = counts

## Observed frequency

In [ ]:
observed = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
observed = pd.concat([observed, observed.sum().to_frame("Total").T])
print(observed)

## Risk

In [ ]:
# Each cell as a share of its own row: A and B are the first row, C and D the second.
risk = table.div(table.sum(axis=1), axis=0)
print()
print(risk)

## Risk difference, risk ratio and odds ratio

In [ ]:
# The intervals are built on the log scale and turned back, because a ratio is skewed:
# 2 and 1/2 are equally far from 1. statsmodels' Table2x2 does exactly that, with
# SE(log RR) = sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d)) and SE(log OR) = sqrt(1/a + 1/b + 1/c + 1/d).
# The B/D risk ratio is the same calculation with the two columns swapped.
# The risk difference has Newcombe's (1998) score interval, the square-and-add of the two
# Wilson score intervals, which Newcombe recommends.
level = 95 / 100
risk_a, risk_b = risk.iloc[0]
risk_c, risk_d = risk.iloc[1]
rows = [["Risk difference", "A-C", risk_a - risk_c,
         *confint_proportions_2indep(a, a + b, c, c + d, method="newcomb", alpha=1 - level)],
        ["Risk difference", "B-D", risk_b - risk_d,
         *confint_proportions_2indep(b, a + b, d, c + d, method="newcomb", alpha=1 - level)]]
if 0 in (a, b, c, d):
    # An empty cell makes a ratio that uses it 0 or infinite and its interval undefined, so
    # those rows stay blank; a ratio whose own cells are all non-zero is still given. In their place: Haldane's correction, which adds 0.5 to every cell (a
    # slightly different table, hence the label), and the conditional maximum-likelihood
    # odds ratio with Cornfield's exact interval, which needs no correction and is 0 or
    # infinite when the data say so.
    print("A cell is 0, so the ratios that use it and their intervals cannot be computed.")
    fourfold = Table2x2(counts, shift_zeros=False)
    swapped = Table2x2(counts[:, ::-1], shift_zeros=False)
    rows += [["Risk ratio", "A/C", *([fourfold.riskratio, *fourfold.riskratio_confint(alpha=1 - level)]
                                     if a and c else [np.nan] * 3)],
             ["Risk ratio", "B/D", *([swapped.riskratio, *swapped.riskratio_confint(alpha=1 - level)]
                                     if b and d else [np.nan] * 3)],
             ["Odds ratio", "AD/BC", np.nan, np.nan, np.nan]]
    haldane = Table2x2(counts + 0.5, shift_zeros=False)
    haldane_swapped = Table2x2(counts[:, ::-1] + 0.5, shift_zeros=False)
    conditional = contingency.odds_ratio(counts.astype(int), kind="conditional")
    interval = conditional.confidence_interval(confidence_level=level)
    rows += [["Risk ratio (Haldane correction)", "A/C", haldane.riskratio,
              *haldane.riskratio_confint(alpha=1 - level)],
             ["Risk ratio (Haldane correction)", "B/D", haldane_swapped.riskratio,
              *haldane_swapped.riskratio_confint(alpha=1 - level)],
             ["Odds ratio (Haldane correction)", "AD/BC", haldane.oddsratio,
              *haldane.oddsratio_confint(alpha=1 - level)],
             ["Odds ratio (conditional MLE)", "AD/BC", conditional.statistic,
              interval.low, interval.high]]
else:
    # shift_zeros=False: statsmodels would otherwise add 0.5 to a table with an empty cell.
    fourfold = Table2x2(counts, shift_zeros=False)
    swapped = Table2x2(counts[:, ::-1], shift_zeros=False)
    rows += [["Risk ratio", "A/C", fourfold.riskratio, *fourfold.riskratio_confint(alpha=1 - level)],
             ["Risk ratio", "B/D", swapped.riskratio, *swapped.riskratio_confint(alpha=1 - level)],
             ["Odds ratio", "AD/BC", fourfold.oddsratio, *fourfold.oddsratio_confint(alpha=1 - level)]]
ratios = pd.DataFrame(rows, columns=["measure", "cells", "value", "lower", "upper"])
print()
print(f"{level:.0%} confidence interval")
print(ratios.to_string(index=False))

## What the result says

Here risk means the share of a group who bought. It is 72.7% with a free sample and 18.2% without. The risk ratio divides the first by the second: it is 4, so shoppers with a sample were 4 times as likely to buy. The risk difference is 0.55, or 55 percentage points.

The confidence interval, the range of plausible values, is wide. For the risk ratio it runs from 1.08 to 14.7. The odds ratio, which compares odds and not shares, is 12, with an interval from 1.58 to 91.1. Do not read it as 12 times as likely.

So the sample helps in this pilot, but with 22 shoppers the true effect could be anything from barely there to very large.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.